# Mirrors Notes 

See pdf 

In [14]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


## Model definition 

We use a MLP with only 4 layers. 

In [4]:
class MLP(nn.Module):
    num_units: int
    
    def setup(self):
        # self.dense1 = nn.Dense(self.num_units)
        # self.dense2 = nn.Dense(self.num_units)
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f + x 
    

class SimpleMLP(nn.Module):
    """
    We ONLY work with "MLP" layers from above; see setup()

    Use sow to save intermediates for computation of preconditioner
    """
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = (
            *[
                MLP(self.num_units, name=f"layer_{i:02}") for i in range(self.num_layers - 1)
            ], 
            MLP(self.num_units, name=f"layer_{self.num_layers-1:02}") # For now, do all the same dimensions 
        )
        
    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}_output', x)
            
        return x


In [16]:
# Model definition 
L = 4 # We use 10 Layers for full runs, 3 layers to debug
n = 1 

n_samples = 1
X = jnp.linspace(0, 1, n_samples).reshape((-1, 1)) + .1

key = jax.random.PRNGKey(0)

model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init(key, jnp.ones((1, n)))['params']
params

{'layer_00': {'dense1': {'kernel': <jax.Array([[-1.5057027]], dtype=float32)>,
   'bias': <jax.Array([0.00563218], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[-0.73806745]], dtype=float32)>,
   'bias': <jax.Array([0.02606239], dtype=float32)>}},
 'layer_01': {'dense1': {'kernel': <jax.Array([[-0.4199463]], dtype=float32)>,
   'bias': <jax.Array([-0.00496414], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[0.9462289]], dtype=float32)>,
   'bias': <jax.Array([-0.01052106], dtype=float32)>}},
 'layer_02': {'dense1': {'kernel': <jax.Array([[-2.0070188]], dtype=float32)>,
   'bias': <jax.Array([0.00379716], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[-0.19703926]], dtype=float32)>,
   'bias': <jax.Array([0.00040974], dtype=float32)>}},
 'layer_03': {'dense1': {'kernel': <jax.Array([[1.5657363]], dtype=float32)>,
   'bias': <jax.Array([0.01429557], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[2.067297]], dtype=float32)>,
   'bias': <jax.Array([-0.00271632], dtype=float32)>}}}

In [8]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    # assert len(x) == 1
    # return jnp.squeeze(layer.apply(params, x))
    return layer.apply(params, x)

K_i = jax.jit(jax.jacrev(apply_layer, argnums=0))
M_i = jax.jit(jax.jacrev(apply_layer, argnums=1))


In [13]:

predictions, intermediates = model.apply({'params':params}, X, mutable=['intermediates'])


## Define the matrices 

In [38]:
# %%timeit # Slightly better
# Initialize dgdu as a tridiagonal matrix with ones down the diagonal
dgdu = np.eye(n_samples * n * (L + 1)) # It's tridiagonal with ones down diagonal; will have to change once scaled up
counter = 0
for l in range(L):
    vals = -jnp.squeeze(M_i({'params': params[f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0]))

    # vals
    for s in range(n_samples):
        dgdu[counter * (n_samples * n) + s * n:counter * (n_samples * n) + s * n + n, 
             counter * (n_samples * n) + (n_samples * n) + s * n:counter * (n_samples * n) + (n_samples * n) + s * n + n] \
                =  vals


    counter += 1
dgdu = dgdu.T
dgduinv = np.linalg.inv(dgdu)

In [39]:
dgdu

array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-2.08828792,  1.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , -0.60678161,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -1.37222954,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -3.98654215,  1.        ]])

In [40]:
dgduinv

array([[ 1.        , -0.        , -0.        , -0.        , -0.        ],
       [ 2.08828792,  1.        , -0.        , -0.        , -0.        ],
       [ 1.26713471,  0.60678161,  1.        , -0.        , -0.        ],
       [ 1.73879968,  0.83264365,  1.37222954,  1.        , -0.        ],
       [ 6.9317982 ,  3.319369  ,  5.4704509 ,  3.98654215,  1.        ]])

In [28]:
# %%timeit
# Number of trainable parameters

dgdt = np.zeros((p, n_samples * n * (L + 1)))
counter = 0
for l in range(L): 
    flattened, _ = jax.tree.flatten(
            K_i({'params': params[f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    # Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
    reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
    
    layer_p = aggregated_array.shape[-1]

    for s in range(n_samples):
        dgdt[counter:counter + layer_p, 
            n_samples * n * l + n_samples * n + s * n:n_samples * n * l + n_samples * n + s * n + n] = aggregated_array[s, :].T
    counter += layer_p
    


In [30]:
dgdt

array([[ 0.        , -0.72277743,  0.        ,  0.        ,  0.        ],
       [ 0.        , -0.07227774,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , -0.14393164,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.93635398,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.21750908,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.10215738,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.18546391,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.02320308,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.24237669,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  1.90743625],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.33051252],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.27807984]])

In [32]:
# Regular way
jacobian = jax.jacobian(model.apply, argnums=0)({'params': params}, X)

flattened, _ = jax.tree.flatten(jacobian)
reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
jacobian = aggregated_array.reshape((X.shape[0] * X.shape[1], -1))
treescope.display(jacobian)



In [59]:
np.linalg.inv(dgdu) @ dgdt.T

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.72277743, -0.07227774,  1.        , -0.14393164,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.43856805, -0.0438568 ,  0.60678161, -0.08733507,  0.93635398,
         0.21750908,  1.        , -0.10215738,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.60181603, -0.0601816 ,  0.83264365, -0.11984377,  1.28489259,
         0.29847238,  1.37222954, -0.14018338, -0.18546391, -0.02320308,
         1.        , -0.24237669,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-2.39916499, -0.23991649,  3.319369  , -0.47776223,  5.12227848,
         1.18987272,  5.4704509 , -0.55884695, -0.73935968, -0.09250004,
         3.98654215, -0.96624487,  1.90743625,  0.33051252,  1.        ,
         0.27807984]])

In [60]:
dgdt.T

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.72277743, -0.07227774,  1.        , -0.14393164,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.93635398,
         0.21750908,  1.        , -0.10215738,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.18546391, -0.02320308,
         1.        , -0.24237669,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  1.90743625,  0.33051252,  1.        ,
         0.27807984]])

In [62]:
dgduinv

array([[ 1.        , -0.        , -0.        , -0.        , -0.        ],
       [ 2.08828792,  1.        , -0.        , -0.        , -0.        ],
       [ 1.26713471,  0.60678161,  1.        , -0.        , -0.        ],
       [ 1.73879968,  0.83264365,  1.37222954,  1.        , -0.        ],
       [ 6.9317982 ,  3.319369  ,  5.4704509 ,  3.98654215,  1.        ]])

In [64]:
jacobian

<jax.Array float32(1, 16) ≈1.1 ±2.2 [≥-2.4, ≤5.5] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>

In [66]:
dgduinv[-1, 1] * dgdt.T[1, 0]

np.float64(-2.3991649851913643)